# indy_mech_extension — no-prompt control
Feed each of the 528 phase-17 response token sequences to Qwen3-8B ALONE (no chat template, no trigger, no query,
nothing prepended) in one teacher-forced pass, and store the residual stream at the same 19 layers and response
positions as `probe_extract.ipynb`. Slot P (prompt-only) does not exist here and is stored as NaN.

In [ ]:
# === CELL 1 — load model + fetch the exact response ids from the HF repo =======================
import torch, json, time, os, hashlib, numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import userdata
from huggingface_hub import hf_hub_download, HfApi
tok = userdata.get("HF_TOKEN"); api = HfApi(token=tok)
REPO = f"{api.whoami()['name']}/indy-mech-extension-qwen3-8b-persona-probes"
ROLL = json.load(open(hf_hub_download(REPO, "rollout_ids.json", repo_type="dataset", token=tok)))
META0 = json.load(open(hf_hub_download(REPO, "features_meta.json", repo_type="dataset", token=tok)))
assert [(r["arm"], r["seed"]) for r in ROLL] == [(r["arm"], r["seed"]) for r in META0["rollouts"]]
MODEL_ID = "Qwen/Qwen3-8B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="cuda:0"); model.eval(); model.requires_grad_(False)
dev = model.device
print(len(ROLL), "rollouts | bos token:", tokenizer.bos_token, "| GPU", torch.cuda.get_device_name(0))


In [ ]:
# === CELL 2 — no-prompt teacher forcing ==========================================================
LAYERS = META0["layers"]; SLOTS = META0["slots"]; KS = META0["ks"]; D = model.config.hidden_size
assert SLOTS[0] == "P"
X = np.full((len(ROLL), len(LAYERS), len(SLOTS), D), np.nan, dtype=np.float16)
t0 = time.time()
with torch.no_grad():
    for i, r in enumerate(ROLL):
        a = r["resp_ids"]; n = len(a)
        if n == 0: continue
        out = model(torch.tensor([a], device=dev), output_hidden_states=True, logits_to_keep=1)
        hs = torch.stack([out.hidden_states[l][0] for l in LAYERS])       # [L, n, D]; position j-1 = state after j tokens
        feats = [torch.full_like(hs[:, 0], float("nan"))]                    # P: no prompt exists
        for k in KS:
            feats.append(hs[:, k - 1] if n >= k else torch.full_like(hs[:, 0], float("nan")))
        feats.append(hs.float().mean(1))                                     # Rmean over all response positions
        feats.append(hs[:, n - 1])                                           # Rlast
        X[i] = torch.stack(feats, 1).float().cpu().numpy().astype(np.float16)
        if i % 100 == 0: print(f"  {i}/{len(ROLL)}  {time.time()-t0:.0f}s")
print(f"done {time.time()-t0:.0f}s | X {X.shape} | nan R1 rows {int(np.isnan(X[:,0,1,0]).sum())}")
np.save("/content/features_noprompt.npy", X)
meta = dict(model=MODEL_ID, layers=LAYERS, slots=SLOTS, ks=KS, hidden=D, condition="no prompt: response ids only, no BOS, no template",
            note="slot P is NaN; R_k = state at response position k-1 (0-indexed), same ids as rollout_ids.json", rollouts=META0["rollouts"])
json.dump(meta, open("/content/features_noprompt_meta.json", "w"))
md5 = hashlib.md5(open("/content/features_noprompt.npy", "rb").read()).hexdigest(); print("md5", md5)
for f in ["features_noprompt.npy", "features_noprompt_meta.json"]:
    api.upload_file(path_or_fileobj=f"/content/{f}", path_in_repo=f"noprompt/{f}", repo_id=REPO, repo_type="dataset"); print("  uploaded noprompt/" + f)
json.dump({"features_noprompt.npy": md5}, open("/content/md5.json", "w"))
api.upload_file(path_or_fileobj="/content/md5.json", path_in_repo="noprompt/md5.json", repo_id=REPO, repo_type="dataset")
print("done", REPO)
